# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/muneeb-khokhar/flyrank-ml-track/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This notebook turns an upstream ranking score into a **human-reviewed, non-production action queue**. It carries forward the final Week-6 validation receipt, maps transparent page archetypes to review actions, adds cost/value planning, defines monitoring and retrain triggers, and exports the exact queue and metrics receipt used by the paper.

The score decides **what a person reviews first**. It never decides what gets edited, published, redirected, deleted, or no-indexed.


## 1. Ranked actions + reason codes

### Evidence carried forward

Week 6 is the validation source, not this action layer. Its final honest configuration used four decision-time features and reported: mean leave-one-client-out Precision@50 of **0.370 across 29 scoreable clients** (range **0.020–0.760**), beating each client's own base rate in **22 of 29** cases at a mean lift of **1.26×**. A separate retrospective, time-ordered top-50 queue contained **28 labelled declines out of 50** against a population base rate of **0.281**. These are different designs and populations, so they remain separate receipts.

The wide client range is why the operational rank is **within each client**. A portfolio-wide rank is not used: score 0.7 for one client is not assumed equivalent to score 0.7 for another.

### Ranking policy

The upstream model score is preserved, then combined with two transparent planning signals:

`review priority = 50% model review score + 30% opportunity proxy + 20% evidence quality`

- **Opportunity proxy** uses only within-client ranks of prior visibility and sessions, plus a small page-one protection flag. It is not revenue or expected uplift.
- **Evidence quality** rewards more days with impressions and enough sessions to support a review.
- The weights are a practical editorial policy, **not learned or validated model weights**. The Week-6 metrics do not describe this re-ranked score.
- `ctr` is already in percentage points: `0.5` means **0.5%**, not 50%.

Reason codes are short, inspectable explanations. Label/future fields (`is_declining_label`, `trend_direction`, `trend_pct`, and last-window outcomes) are excluded from both actions and exported rows.


In [1]:
import hashlib
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_colwidth", 80)
pd.set_option("display.width", 180)

def find_project_root():
    start = Path.cwd().resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "data/raw/content_refresh_anonymized.csv").exists():
            return candidate
    raise FileNotFoundError("Run this notebook from inside the flyrank-ml-track repository.")

PROJECT_ROOT = find_project_root()
WORK_OUTPUTS = PROJECT_ROOT / "work/outputs"
WORK_OUTPUTS.mkdir(parents=True, exist_ok=True)

# Aggregate receipt from the final printed claim in the executed Week-6 notebook.
VALIDATION_RECEIPT = {
    "source_notebook": "work/notebooks/w06_validation_audit.ipynb",
    "decision_date": "2026-03-31",
    "honest_features": ["imp_prev30", "clk_prev30", "pos_prev30", "days_active_prev30"],
    "loco_scoreable_clients": 29,
    "loco_total_clients": 42,
    "loco_mean_precision_at_50": 0.370,
    "loco_median_precision_at_50": 0.300,
    "loco_range": [0.020, 0.760],
    "loco_clients_beating_own_base_rate": 22,
    "loco_mean_lift_over_own_base_rate": 1.26,
    "time_aware_week5_frame_precision_at_50": 0.260,
    "time_aware_week5_frame_base_rate": 0.174,
    "time_aware_unseen_client_mean_precision_at_50": 0.171,
    "retrospective_unfiltered_queue_precision_at_50": 0.560,
    "retrospective_unfiltered_queue_base_rate": 0.281,
    "freshness_rows_not_knowable_at_decision_time_pct": 80.1,
}

# Verify the hand-carried values against Week 6 so they cannot silently drift.
w06_path = PROJECT_ROOT / VALIDATION_RECEIPT["source_notebook"]
w06 = json.loads(w06_path.read_text(encoding="utf-8"))
output_parts = []
for cell in w06.get("cells", []):
    for output in cell.get("outputs", []):
        if output.get("output_type") == "stream":
            value = output.get("text", "")
        else:
            value = output.get("data", {}).get("text/plain", "")
        output_parts.append("".join(value) if isinstance(value, list) else str(value))
w06_output_text = "\n".join(output_parts)
receipt_snippets = [
    "mean P@50   0.370",
    "beat their own base rate: 22 of 29 clients",
    "mean lift over each client's own base rate: 1.26x",
    "time-aware Feb -> Mar",
    "Top-50 queue: 28 of 50 really declined (P@50 0.560)",
    "days_since_update negative: 65,211 of 81,446 rows (80.1%)",
]
missing_receipts = [text for text in receipt_snippets if text not in w06_output_text]
assert not missing_receipts, f"Week-6 receipt changed or is unexecuted: {missing_receipts}"

# Prefer the full locally regenerated queue. A clean clone uses the committed top-200 sample.
full_source = PROJECT_ROOT / "outputs/refresh_queue.csv"
sample_source = PROJECT_ROOT / "outputs/refresh_queue_sample.csv"
if full_source.exists():
    SOURCE_PATH = full_source
    SOURCE_MODE = "full_regenerated_queue"
else:
    SOURCE_PATH = sample_source
    SOURCE_MODE = "committed_top_200_interface_sample"
    warnings.warn(
        "Full outputs/refresh_queue.csv is absent; executing on the committed top-200 "
        "interface sample. Generate the full upstream queue and Run All before making "
        "population-wide claims."
    )

upstream = pd.read_csv(SOURCE_PATH)
required = {
    "content_id", "client_id", "best_model_probability", "impressions_90d",
    "sessions_90d", "avg_position", "ctr", "content_age_days",
    "days_since_last_update", "word_count",
}
missing = sorted(required - set(upstream.columns))
assert not missing, f"Upstream queue is missing required columns: {missing}"
assert upstream["content_id"].is_unique, "Expected one upstream row per content item."

# Add review context from the public anonymised starter slice. None are labels.
context_columns = ["content_id", "engagement_rate", "scroll_rate", "days_with_impressions"]
context = pd.read_csv(
    PROJECT_ROOT / "data/raw/content_refresh_anonymized.csv",
    usecols=context_columns,
)
assert context["content_id"].is_unique
upstream = upstream.merge(context, on="content_id", how="left", validate="one_to_one")

# Explicit allow-list: label, trend, title, URL, domain and query fields never enter action_base.
allowed_columns = [
    "content_id", "client_id", "best_model_probability", "impressions_90d",
    "sessions_90d", "avg_position", "ctr", "content_age_days",
    "days_since_last_update", "word_count", "engagement_rate",
    "scroll_rate", "days_with_impressions",
]
action_base = upstream[allowed_columns].copy()
numeric_columns = [c for c in allowed_columns if c not in {"content_id", "client_id"}]
missing_rate_reference = action_base[numeric_columns].isna().mean().round(4).to_dict()
for column in numeric_columns:
    action_base[column] = pd.to_numeric(action_base[column], errors="coerce").fillna(0)

print("Validated receipt: Week-6 outputs matched all required snippets.")
print(f"Queue source: {SOURCE_PATH.relative_to(PROJECT_ROOT)}")
print(f"Source mode: {SOURCE_MODE}")
print(f"Rows: {len(action_base):,} | clients: {action_base.client_id.nunique()}")
print("Rate-unit check: ctr/engagement/scroll remain percentage points (0.5 means 0.5%).")


Validated receipt: Week-6 outputs matched all required snippets.
Queue source: outputs\refresh_queue.csv
Source mode: full_regenerated_queue
Rows: 30,000 | clients: 32
Rate-unit check: ctr/engagement/scroll remain percentage points (0.5 means 0.5%).


In [2]:
# Only these fields affect rank. Freshness and word count explain the review path,
# but do not move the priority score.
PRIORITY_INPUTS = {
    "best_model_probability", "impressions_90d", "sessions_90d",
    "avg_position", "days_with_impressions",
}
BANNED_FOR_ACTIONS = {
    "is_declining_label", "trend_direction", "trend_pct",
    "impressions_last_30d", "clicks_last_30d", "sessions_last_30d",
}
assert PRIORITY_INPUTS.isdisjoint(BANNED_FOR_ACTIONS)

queue = action_base.copy()
queue["model_review_score"] = queue["best_model_probability"].clip(0, 1)

def within_client_percentile(series):
    return np.log1p(series.clip(lower=0)).rank(method="average", pct=True)

queue["impression_value_pct"] = queue.groupby("client_id")["impressions_90d"].transform(within_client_percentile)
queue["session_value_pct"] = queue.groupby("client_id")["sessions_90d"].transform(within_client_percentile)
queue["page_one_flag"] = ((queue["avg_position"] > 0) & (queue["avg_position"] <= 10)).astype(int)
queue["opportunity_proxy"] = (
    0.70 * queue["impression_value_pct"]
    + 0.20 * queue["session_value_pct"]
    + 0.10 * queue["page_one_flag"]
).clip(0, 1)
queue["evidence_quality"] = (
    0.55 * (queue["days_with_impressions"].clip(0, 90) / 90)
    + 0.45 * (queue["sessions_90d"].clip(0, 30) / 30)
).clip(0, 1)
queue["review_priority_score"] = (100 * (
    0.50 * queue["model_review_score"]
    + 0.30 * queue["opportunity_proxy"]
    + 0.20 * queue["evidence_quality"]
)).round(1)

# Non-label flags for reason codes and archetype-to-action mapping.
queue["limited_signal"] = (queue["impressions_90d"] < 100) | (queue["days_with_impressions"] < 10)
queue["page_one_winner"] = queue["avg_position"].between(0.000001, 10) & (queue["impressions_90d"] >= 1000)
queue["stale_visible"] = (queue["days_since_last_update"] >= 180) & (queue["impressions_90d"] >= 500)
queue["thin_visible"] = queue["word_count"].between(0.000001, 1199.999) & (queue["impressions_90d"] >= 250)
queue["ctr_opportunity"] = (
    (queue["impressions_90d"] >= 500)
    & queue["avg_position"].between(0.000001, 20)
    & (queue["ctr"] < 0.5)
)
queue["engagement_gap"] = (
    (queue["sessions_90d"] >= 30)
    & (
        queue["engagement_rate"].between(0.000001, 29.999)
        | queue["scroll_rate"].between(0.000001, 29.999)
    )
)

def make_reason_codes(row):
    codes = ["model_ranked_candidate"]
    if row["model_review_score"] >= 0.65:
        codes.append("high_model_review_score")
    if row["impressions_90d"] >= 1000:
        codes.append("visible_demand")
    if row["page_one_winner"]:
        codes.append("page_one_to_protect")
    if row["stale_visible"]:
        codes.append("stale_180d_and_visible")
    if row["thin_visible"]:
        codes.append("thin_under_1200_words")
    if row["ctr_opportunity"]:
        codes.append("low_ctr_visible_page")
    if row["engagement_gap"]:
        codes.append("weak_engagement_signal")
    if row["limited_signal"]:
        codes.append("limited_evidence")
    return "|".join(codes)

def choose_archetype(row):
    if row["limited_signal"]:
        return "limited_signal"
    if row["page_one_winner"]:
        return "protect_winner"
    if row["stale_visible"]:
        return "stale_visible"
    if row["thin_visible"]:
        return "thin_visible"
    if row["ctr_opportunity"]:
        return "ctr_opportunity"
    if row["engagement_gap"]:
        return "engagement_gap"
    return "decline_risk_review"

ACTION_MAP = {
    "limited_signal": {
        "action": "monitor_and_collect_data", "minutes": 5,
        "human_gate": "Confirm tracking and wait for enough observations; do not edit from this score.",
    },
    "protect_winner": {
        "action": "diagnose_before_editing", "minutes": 20,
        "human_gate": "Check indexation, cannibalisation, SERP intent and seasonality before touching a winner.",
    },
    "stale_visible": {
        "action": "manual_refresh_review", "minutes": 30,
        "human_gate": "Verify facts, intent and decay; staleness alone is not permission to refresh.",
    },
    "thin_visible": {
        "action": "review_depth_and_intent", "minutes": 35,
        "human_gate": "Check whether more depth serves intent; never add words only to hit a count.",
    },
    "ctr_opportunity": {
        "action": "review_serp_snippet_and_intent", "minutes": 15,
        "human_gate": "Compare query intent and live SERP; a low aggregate CTR is not a title diagnosis.",
    },
    "engagement_gap": {
        "action": "review_content_experience", "minutes": 25,
        "human_gate": "Check analytics coverage, intent match, structure and page experience.",
    },
    "decline_risk_review": {
        "action": "manual_diagnostic_review", "minutes": 20,
        "human_gate": "Inspect technical, demand, competitor and content explanations before choosing an edit.",
    },
}

queue["reason_codes"] = queue.apply(make_reason_codes, axis=1)
queue["archetype"] = queue.apply(choose_archetype, axis=1)
queue["recommended_action"] = queue["archetype"].map(lambda x: ACTION_MAP[x]["action"])
queue["planning_effort_minutes"] = queue["archetype"].map(lambda x: ACTION_MAP[x]["minutes"])
queue["human_review_gate"] = queue["archetype"].map(lambda x: ACTION_MAP[x]["human_gate"])
queue["review_confidence"] = np.select(
    [
        queue["limited_signal"],
        (queue["model_review_score"] >= 0.65) & (queue["evidence_quality"] >= 0.70),
        (queue["model_review_score"] >= 0.50) & (queue["evidence_quality"] >= 0.45),
    ],
    ["limited", "high_signal_completeness", "medium_signal_completeness"],
    default="low_signal_completeness",
)
queue["cost_value_index"] = (queue["review_priority_score"] / queue["planning_effort_minutes"]).round(2)
queue["automation_status"] = "human_decision_required"

# Deterministic rank within client. Cost/value is visible but does not override this rank.
queue = queue.sort_values(
    ["client_id", "review_priority_score", "model_review_score", "impressions_90d", "content_id"],
    ascending=[True, False, False, False, True],
    kind="mergesort",
).reset_index(drop=True)
queue["client_queue_rank"] = queue.groupby("client_id").cumcount() + 1
queue["review_band"] = np.select(
    [queue["client_queue_rank"] <= 5, queue["client_queue_rank"] <= 15],
    ["review_now", "review_next"],
    default="monitor_later",
)

export_columns = [
    "client_queue_rank", "content_id", "client_id", "review_band",
    "review_priority_score", "model_review_score", "opportunity_proxy",
    "evidence_quality", "cost_value_index", "archetype", "recommended_action",
    "reason_codes", "review_confidence", "planning_effort_minutes",
    "human_review_gate", "automation_status", "impressions_90d", "sessions_90d",
    "avg_position", "ctr", "content_age_days", "days_since_last_update",
    "word_count", "days_with_impressions",
]
ranked_queue = queue[export_columns].copy()

action_table = pd.DataFrame([
    {
        "archetype": archetype,
        "recommended_action": policy["action"],
        "planning_minutes": policy["minutes"],
        "human_gate": policy["human_gate"],
    }
    for archetype, policy in ACTION_MAP.items()
])
print("Archetype -> action map (minutes are assumptions, not observed labour times):")
print(action_table.to_string(index=False))
print("\nTop three candidates per client (first 15 rows shown):")
preview = ranked_queue.groupby("client_id", sort=False).head(3).head(15)
print(preview[[
    "client_queue_rank", "content_id", "client_id", "review_priority_score",
    "archetype", "recommended_action", "reason_codes",
]].to_string(index=False))


Archetype -> action map (minutes are assumptions, not observed labour times):
          archetype             recommended_action  planning_minutes                                                                               human_gate
     limited_signal       monitor_and_collect_data                 5          Confirm tracking and wait for enough observations; do not edit from this score.
     protect_winner        diagnose_before_editing                20 Check indexation, cannibalisation, SERP intent and seasonality before touching a winner.
      stale_visible          manual_refresh_review                30            Verify facts, intent and decay; staleness alone is not permission to refresh.
       thin_visible        review_depth_and_intent                35             Check whether more depth serves intent; never add words only to hit a count.
    ctr_opportunity review_serp_snippet_and_intent                15        Compare query intent and live SERP; a low aggregate CTR 

## 2. Intended use and limits

### Intended use

A content editor or SEO strategist uses this queue once per review cycle to choose a small set of pages to inspect. The correct workflow is **rank → diagnose → decide → log**, not rank → auto-edit. One row is one pseudonymised content item; the useful rank is `client_queue_rank`. `review_now` means top five candidates for that client, not five guaranteed problems.

### Limits that travel with every row

1. **Decision support, not treatment effect.** The data did not assign refreshes, so it cannot show that an edit will recover traffic.
2. **Scores are not calibrated across clients.** Week 6 measured a 0.020–0.760 client range. Compare pages within a client; do not auction editorial time across clients from the raw score.
3. **Small and new clients are outside the strongest evidence.** Thirteen of 42 clients in the final honest LOCO frame were not scoreable at Precision@50; the time-aware unseen-client mean was 0.171 on its filtered frame. New clients start in shadow mode.
4. **The label is a threshold.** A page at 0.79 of prior impressions is labelled declining and one at 0.81 is not. Near-boundary errors are partly definitional.
5. **No seasonality, cause or revenue.** Opportunity is a traffic/value proxy. It does not price revenue, brand risk, legal risk, or editor effort.
6. **Public-clone fallback.** If the ignored full queue is absent, this notebook uses the committed top-200 sample. That proves the action interface runs; it does not support portfolio-wide action shares. The metrics receipt records the source mode.

### The decay/refresh insight

Freshness is a **review context**, not an automatic treatment. Week 6 found that historical `days_since_update` was not knowable at the decision date for 80.1% of its modelled rows, so freshness is not added to this priority formula. It only selects the human diagnostic path:

- stale + visible → inspect facts, intent and decay; refresh only if the page itself supports it;
- page-one winner → protect and diagnose before editing;
- thin + visible → test whether depth serves intent, not whether it reaches a word count;
- limited evidence → monitor and repair measurement before editing.


In [3]:
print(f"Input scope: {SOURCE_MODE}")
print(f"Rows exported: {len(ranked_queue):,} | clients: {ranked_queue.client_id.nunique()}")
print("\nArchetype counts (descriptive of this input only):")
print(ranked_queue["archetype"].value_counts().to_string())
print("\nRecommended action counts:")
print(ranked_queue["recommended_action"].value_counts().to_string())
print("\nReview bands:")
print(ranked_queue["review_band"].value_counts().to_string())

review_now = ranked_queue[ranked_queue["review_band"] == "review_now"]
planning_minutes = int(review_now["planning_effort_minutes"].sum())
print(f"\nCapacity illustration: {len(review_now)} review-now rows = {planning_minutes:,} assumed review minutes.")
print("This is workload planning, not measured cost.")
print("No expected traffic or revenue uplift is calculated because this study has no treatment effect.")


Input scope: full_regenerated_queue
Rows exported: 30,000 | clients: 32

Archetype counts (descriptive of this input only):
archetype
decline_risk_review    8443
limited_signal         7997
protect_winner         6591
ctr_opportunity        4560
engagement_gap         2324
thin_visible             69
stale_visible            16

Recommended action counts:
recommended_action
manual_diagnostic_review          8443
monitor_and_collect_data          7997
diagnose_before_editing           6591
review_serp_snippet_and_intent    4560
review_content_experience         2324
review_depth_and_intent             69
manual_refresh_review               16

Review bands:
review_band
monitor_later    29532
review_next        310
review_now         158

Capacity illustration: 158 review-now rows = 2,980 assumed review minutes.
This is workload planning, not measured cost.
No expected traffic or revenue uplift is calculated because this study has no treatment effect.


## 3. Human review + the no-go list

### Required review before any action

For every `review_now` row, a person must check:

1. **Measurement:** tracking outage, migration, indexation change, or missing data?
2. **Demand:** seasonality, news cycle, campaign ending, or market demand shift?
3. **Search context:** live SERP, query intent, competitor change, cannibalisation, or SERP-feature change?
4. **Page context:** factual age, quality, structure, internal links, conversion purpose, brand/legal sensitivity?
5. **Decision log:** accept, defer, monitor, or reject the recommendation; record a short reason and owner.

### What must **not** be automated

- publishing or rewriting page copy; changing titles/meta descriptions; adding words;
- redirecting, deleting, pruning, canonicalising, or no-indexing pages;
- diagnosing why traffic changed from the score alone;
- acting on a stale flag alone, or treating score as expected uplift/revenue;
- applying this model to a new client without shadow validation;
- exposing titles, URLs, domains, client names, or raw queries in public outputs;
- automatic retraining, threshold changes, model promotion, or deployment.

The only safe automation here is mechanical: calculate scores, generate a queue, attach reason codes, and collect reviewer decisions.


In [4]:
human_policy = pd.DataFrame([
    ("score and rank pseudonymous rows", "allowed", "Mechanical decision support only"),
    ("attach transparent reason codes", "allowed", "Codes explain signals, not causes"),
    ("record reviewer accept/defer/reject", "allowed", "Creates feedback for later evaluation"),
    ("rewrite or publish content", "no-go", "Requires editorial approval and page context"),
    ("redirect, delete, canonicalise or no-index", "no-go", "High-impact and difficult to reverse"),
    ("diagnose cause or promise recovery", "no-go", "Observational data cannot carry the claim"),
    ("retrain or deploy automatically", "no-go", "Requires matured labels and human approval"),
], columns=["operation", "policy", "reason"])
print(human_policy.to_string(index=False))

private_fragments = ("url", "domain", "query", "keyword", "title", "client_name")
private_columns = [
    column for column in ranked_queue.columns
    if any(fragment in column.lower() for fragment in private_fragments)
]
label_columns = sorted(set(ranked_queue.columns) & BANNED_FOR_ACTIONS)
print(f"\nPrivate/context columns in public queue: {private_columns or 'none'}")
print(f"Label/future columns in public queue: {label_columns or 'none'}")
human_rows = int((ranked_queue["automation_status"] == "human_decision_required").sum())
print(f"Rows requiring a human decision: {human_rows:,} of {len(ranked_queue):,}")


                                 operation  policy                                       reason
          score and rank pseudonymous rows allowed             Mechanical decision support only
           attach transparent reason codes allowed            Codes explain signals, not causes
       record reviewer accept/defer/reject allowed        Creates feedback for later evaluation
                rewrite or publish content   no-go Requires editorial approval and page context
redirect, delete, canonicalise or no-index   no-go         High-impact and difficult to reverse
        diagnose cause or promise recovery   no-go    Observational data cannot carry the claim
           retrain or deploy automatically   no-go   Requires matured labels and human approval

Private/context columns in public queue: none
Label/future columns in public queue: none
Rows requiring a human decision: 30,000 of 30,000


## 4. Monitoring / retrain triggers

Monitoring stays light because this is not a production service. Each run saves a small aggregate reference in the metrics JSON; no row-level data is committed.

| Check | Provisional trigger | Human response |
|---|---|---|
| Data contract | any missing required field, duplicate content ID, wrong rate unit, or future/label field entering actions | stop the run; repair before scoring |
| Coverage drift | required-feature missingness rises by >5 percentage points | inspect extraction/tracking; do not silently fill and proceed |
| Score/action drift | score PSI >0.20 or any action share moves >20 percentage points | audit source mix, windows and mapping |
| Outcome quality | within-client Precision@50 is at or below that client's base rate for two matured windows | pause that client's model ranking; compare with the rule and retrain only if justified |
| Reviewer feedback | >40% of `review_now` rows rejected for the same reason in two cycles | revise reason/action rule; do not blame reviewers |
| New client | fewer than 50 mature rows or only one label class | shadow mode; no performance claim |
| Age | 90 days since last approved evaluation, or a material schema/window change | re-evaluate before deciding whether to retrain |

The numeric drift thresholds are starting guardrails, not discovered constants. A trigger opens a review; it does not automatically retrain or deploy anything.


In [5]:
MONITORING_TRIGGERS = [
    {"check": "data_contract", "trigger": "any violation", "response": "stop and repair", "automatic_retrain": False},
    {"check": "missingness_shift", "trigger": "> 0.05 absolute increase", "response": "inspect extraction and tracking", "automatic_retrain": False},
    {"check": "score_or_action_drift", "trigger": "PSI > 0.20 or action share shift > 0.20", "response": "audit data mix, windows and mapping", "automatic_retrain": False},
    {"check": "matured_outcome_quality", "trigger": "client P@50 <= own base rate for 2 windows", "response": "pause client ranking and compare with baseline", "automatic_retrain": False},
    {"check": "reviewer_override", "trigger": "> 0.40 same-reason rejection for 2 cycles", "response": "review reason and action mapping", "automatic_retrain": False},
    {"check": "new_or_small_client", "trigger": "< 50 mature rows or one class", "response": "shadow mode", "automatic_retrain": False},
]

monitor_reference = {
    "source_mode": SOURCE_MODE,
    "rows": int(len(ranked_queue)),
    "clients": int(ranked_queue["client_id"].nunique()),
    "model_review_score_quantiles": {
        str(q): round(float(ranked_queue["model_review_score"].quantile(q)), 4)
        for q in [0.1, 0.5, 0.9]
    },
    "review_priority_quantiles": {
        str(q): round(float(ranked_queue["review_priority_score"].quantile(q)), 2)
        for q in [0.1, 0.5, 0.9]
    },
    "action_share": {
        key: round(float(value), 4)
        for key, value in ranked_queue["recommended_action"].value_counts(normalize=True).items()
    },
    "archetype_share": {
        key: round(float(value), 4)
        for key, value in ranked_queue["archetype"].value_counts(normalize=True).items()
    },
    "missing_rate_before_fill": missing_rate_reference,
}

print(pd.DataFrame(MONITORING_TRIGGERS).to_string(index=False))
print("\nCurrent aggregate reference (descriptive, not a performance result):")
print(json.dumps(monitor_reference, indent=2, sort_keys=True))


                  check                                    trigger                                       response  automatic_retrain
          data_contract                              any violation                                stop and repair              False
      missingness_shift                   > 0.05 absolute increase                inspect extraction and tracking              False
  score_or_action_drift    PSI > 0.20 or action share shift > 0.20            audit data mix, windows and mapping              False
matured_outcome_quality client P@50 <= own base rate for 2 windows pause client ranking and compare with baseline              False
      reviewer_override  > 0.40 same-reason rejection for 2 cycles               review reason and action mapping              False
    new_or_small_client              < 50 mature rows or one class                                    shadow mode              False

Current aggregate reference (descriptive, not a performance result):

## 5. Exports for the paper

This notebook writes two files:

- `work/outputs/w07_ranked_action_queue.csv` — row-level pseudonymous queue; regenerated locally and intentionally ignored by git.
- `work/outputs/w07_playbook_metrics.json` — small aggregate receipt; safe to commit with the notebook. It records input mode, hashes, counts, validation metrics, action shares, monitoring reference, and limitations.

No figure is exported here. The paper can make its final figure from the queue and trace every stated number to the committed JSON receipt, without committing another row-level data file.


In [6]:
QUEUE_OUTPUT = WORK_OUTPUTS / "w07_ranked_action_queue.csv"
METRICS_OUTPUT = WORK_OUTPUTS / "w07_playbook_metrics.json"

ranked_queue.to_csv(QUEUE_OUTPUT, index=False)
queue_sha256 = hashlib.sha256(QUEUE_OUTPUT.read_bytes()).hexdigest()
source_sha256 = hashlib.sha256(SOURCE_PATH.read_bytes()).hexdigest()

NO_GO = [
    "no automatic rewrite or publish",
    "no automatic redirect, delete, canonical or no-index",
    "no cause or recovery claim",
    "no cross-client score comparison",
    "no new-client use outside shadow mode",
    "no client names, URLs, domains, titles or raw queries in public exports",
    "no automatic retrain, threshold change, promotion or deployment",
]

metrics_receipt = {
    "artifact": "w07_content_action_playbook",
    "intended_use": "within-client human review prioritisation; non-production",
    "source_path": str(SOURCE_PATH.relative_to(PROJECT_ROOT)),
    "source_mode": SOURCE_MODE,
    "source_sha256": source_sha256,
    "queue_path": str(QUEUE_OUTPUT.relative_to(PROJECT_ROOT)),
    "queue_sha256": queue_sha256,
    "rows_exported": int(len(ranked_queue)),
    "clients": int(ranked_queue["client_id"].nunique()),
    "priority_formula": {
        "model_review_score": 0.50,
        "within_client_opportunity_proxy": 0.30,
        "evidence_quality": 0.20,
        "status": "editorial planning heuristic; not validated model weights",
    },
    "priority_inputs": sorted(PRIORITY_INPUTS),
    "label_or_future_inputs_used": [],
    "validation_receipt": VALIDATION_RECEIPT,
    "action_counts": {key: int(value) for key, value in ranked_queue["recommended_action"].value_counts().items()},
    "archetype_counts": {key: int(value) for key, value in ranked_queue["archetype"].value_counts().items()},
    "review_band_counts": {key: int(value) for key, value in ranked_queue["review_band"].value_counts().items()},
    "review_now_planning_minutes": planning_minutes,
    "planning_minutes_status": "assumptions for capacity only; not observed cost",
    "monitor_reference": monitor_reference,
    "monitoring_triggers": MONITORING_TRIGGERS,
    "no_go": NO_GO,
    "not_claimed": [
        "that a flagged page is certainly declining",
        "that refreshing a flagged page causes recovery",
        "that the planning score is calibrated across clients",
        "that the opportunity proxy is revenue or expected uplift",
        "that Week-6 validation metrics apply to the heuristic re-rank",
    ],
}
METRICS_OUTPUT.write_text(json.dumps(metrics_receipt, indent=2, sort_keys=True), encoding="utf-8")

print(f"Wrote queue: {QUEUE_OUTPUT.relative_to(PROJECT_ROOT)} ({QUEUE_OUTPUT.stat().st_size:,} bytes)")
print(f"  SHA-256: {queue_sha256}")
print(f"Wrote receipt: {METRICS_OUTPUT.relative_to(PROJECT_ROOT)} ({METRICS_OUTPUT.stat().st_size:,} bytes)")
print("Commit the notebook and metrics JSON. Leave the queue CSV ignored; Run All regenerates it.")


Wrote queue: work\outputs\w07_ranked_action_queue.csv (11,675,704 bytes)
  SHA-256: ddf09867446357240b6def03b6ce927e418b2474e2ae262c3fc8c8d4114f452c
Wrote receipt: work\outputs\w07_playbook_metrics.json (5,790 bytes)
Commit the notebook and metrics JSON. Leave the queue CSV ignored; Run All regenerates it.


## 6. Self-check

- [x] Ranked actions have transparent reason codes and archetype → action mapping.
- [x] Intended user, use, limits, decay/refresh insight, cost/value assumptions, and no-go cases are stated.
- [x] Every row requires human review; no content mutation is automated.
- [x] Monitoring and retrain triggers open a human review and never auto-deploy.
- [x] Queue CSV and aggregate metrics JSON are regenerated under `work/outputs/`.
- [x] No client names, URLs, domains, titles, raw queries, labels, or future outcomes are exported.
- [x] Claims use observed/measured/directional/decision-support language and keep designs separate.
- [x] Notebook runs top to bottom with no errors.


In [7]:
checks = {}
checks["exports_exist"] = QUEUE_OUTPUT.exists() and METRICS_OUTPUT.exists()
checks["one_row_per_content"] = ranked_queue["content_id"].is_unique
checks["rank_unique_within_client"] = not ranked_queue.duplicated(["client_id", "client_queue_rank"]).any()
checks["actions_complete"] = ranked_queue[["recommended_action", "reason_codes", "human_review_gate"]].notna().all().all()
checks["human_required_for_every_row"] = ranked_queue["automation_status"].eq("human_decision_required").all()
checks["no_label_or_future_columns"] = not bool(set(ranked_queue.columns) & BANNED_FOR_ACTIONS)
checks["no_private_context_columns"] = not private_columns
checks["priority_inputs_are_allowed"] = PRIORITY_INPUTS.isdisjoint(BANNED_FOR_ACTIONS)
checks["week6_receipt_verified"] = not missing_receipts
checks["queue_hash_matches_receipt"] = (
    hashlib.sha256(QUEUE_OUTPUT.read_bytes()).hexdigest() == metrics_receipt["queue_sha256"]
)
checks["no_automatic_retrain_trigger"] = all(
    not item["automatic_retrain"] for item in MONITORING_TRIGGERS
)

failed = [name for name, passed in checks.items() if not passed]
for name, passed in checks.items():
    print(f"{'PASS' if passed else 'FAIL'}  {name}")
assert not failed, f"Self-check failed: {failed}"
print(f"\nAll {len(checks)} checks passed. Executed source mode: {SOURCE_MODE}.")


PASS  exports_exist
PASS  one_row_per_content
PASS  rank_unique_within_client
PASS  actions_complete
PASS  human_required_for_every_row
PASS  no_label_or_future_columns
PASS  no_private_context_columns
PASS  priority_inputs_are_allowed
PASS  week6_receipt_verified
PASS  queue_hash_matches_receipt
PASS  no_automatic_retrain_trigger

All 11 checks passed. Executed source mode: full_regenerated_queue.
